Inputs: Training Set (Data, Labels)

Outputs: Metrics: Loss and Accuracy

In [ ]:
# sync with github so my imports are here
!git clone https://github.com/ellylai/10707-project.git
%cd 10707-project

!pip install -q transformers datasets scikit-learn

import sys
sys.path.append("/content/10707-project")

In [5]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Adding specific directories to sys.path since they are not recognized as Python packages (missing __init__.py files).
# The parent directory '/content/10707-project' was already added in a previous cell.
import sys
if "/content/10707-project/datasets" not in sys.path:
    sys.path.append("/content/10707-project/datasets")
if "/content/10707-project/utils" not in sys.path:
    sys.path.append("/content/10707-project/utils")

# Now import modules directly by their filenames
from download_datasets import *
from training_utils import *
from create_dataset_splits import *
from get_waveform_dataset import *

## Unzip and Unload `.wav` Files

In [ ]:
# 1. Mount Drive to access the zips
from google.colab import drive
import os

# 1. Mount your Drive
drive.mount('/content/drive')

# 2. Path to the shortcut you just created
# (Replace 'XMAD-dataset' with whatever you named the shortcut)
!ls /content/drive/MyDrive/
dataset_path = '/content/drive/MyDrive/XMAD-dataset'

# 3. Verify you can see the zips
print("Files in shared folder:")
!ls "{dataset_path}"

!mkdir -p /content/xmad_local
# languages = ['en.zip', 'zh-cn.zip', 'es.zip'] # Add more as needed: 'ar.zip', 'de.zip', etc.
languages = ['ar.zip', 'de.zip', 'ro.zip', 'ru.zip'] # + ['en.zip', 'zh-cn.zip', 'es.zip']

for lang in languages:
    zip_path = os.path.join(dataset_path, lang)
    if os.path.exists(zip_path):
        print(f"Unzipping {lang}...")
        # -n skips files that already exist to save time if you re-run
        !unzip -nq "{zip_path}" -d /content/xmad_local/
    else:
        print(f"Warning: {lang} not found in {dataset_path}")

print("Unzip process complete.")

In [ ]:
!pip install awscli
!aws s3api get-bucket-location --bucket 10707-project

In [ ]:
# 1. Increase the number of parallel threads (Default is 10, set to 50+)
!aws configure set default.s3.max_concurrent_requests 100

# 2. Lower the threshold for multipart uploads (Good for small-medium files)
!aws configure set default.s3.multipart_threshold 8MB

# 3. Now run the sync again
!aws s3 sync /content/xmad_local/ s3://10707-project/xmad_bench/ --size-only --only-show-errors

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Root directory where your languages are unzipped
root_dir = "/content/xmad_local"
all_data = []

# Walk through all directories to find meta.csv files
for path in Path(root_dir).rglob('meta.csv'):
    # Read the individual metadata file
    df = pd.read_csv(path)

    # Identify the dataset source from the folder structure
    parent_folder = path.parent.name

    if "commonvoice" in parent_folder.lower():
        # CommonVoice contains the internal 'train' and 'test' labels
        # We use 'train' for training and 'test' as our Validation set
        df['split'] = df['split'].map({'train': 'train', 'test': 'val'})
    else:
        # AISHELL-3, M-AILABS, etc., are strictly for Cross-Domain Testing
        df['split'] = 'test'

    # Convert relative 'file' paths to absolute paths for the DataLoader
    # This ensures your training_pipeline.ipynb can find the .wav files
    df['file'] = df['file'].apply(lambda x: os.path.join(path.parent, x))

    all_data.append(df)

# Combine all parsed metadata into one master dataframe
if all_data:
    master_df = pd.concat(all_data, ignore_index=True)

    # Ensure the data directory exists in your cloned repo
    os.makedirs("/content/10707-project/data", exist_ok=True)

    # Save the manifest to the path expected by your notebook
    output_path = "/content/10707-project/data/speechfake_splits.csv"
    master_df.to_csv(output_path, index=False)

    print(f"Successfully created: {output_path}")

    # upload to s3 bucket
    s3_dest = "s3://10707-project/xmad_bench/metadata/speechfake_splits.csv"
    !aws s3 cp {output_path} {s3_dest}
    print(f"Successfully uploaded to S3: {s3_dest}")

    print("Split Distribution:")
    print(master_df['split'].value_counts())
else:
    print("No meta.csv files found. Ensure the unzip process finished correctly.")

## Download `.wav` Files from AWS S3

In [ ]:
# to download from AWS S3
# 1. Install AWS CLI (Not pre-installed on Colab)
!pip install -q awscli

# 2. OPTIONAL: If you saved your credentials in a file on Drive, load them here
# Otherwise, run !aws configure to log in again

# 3. Optimize for high-speed download
!aws configure set default.s3.max_concurrent_requests 100
!aws configure set default.s3.multipart_threshold 8MB

# 4. Sync from S3 to Local Colab Disk
# We use --size-only to skip unnecessary checks and speed up the process
import os
os.makedirs("/content/xmad_local", exist_ok=True)

print("Starting high-speed download from S3...")
!aws s3 sync s3://10707-project/xmad_bench/ /content/xmad_local/ --size-only --only-show-errors

print("Download complete. Checking local storage:")
!df -h /content

## Create Dataset & Train

In [ ]:
import torch
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class XMADDataset(Dataset):
    def __init__(self, df, target_sr=16000, max_seconds=4.0):
        self.df = df
        self.target_sr = target_sr
        self.max_samples = int(target_sr * max_seconds)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file']
        label = 1 if row['label'] == 'spoof' else 0 # Adjust based on your CSV label names

        # Load audio
        waveform, sr = torchaudio.load(file_path)

        # Resample if necessary
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or Trim to fixed length (crucial for batching)
        if waveform.shape[1] > self.max_samples:
            waveform = waveform[:, :self.max_samples]
        else:
            padding = self.max_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))

        return waveform.squeeze(0), torch.tensor(label)

In [ ]:
# Load your master manifest
master_df = pd.read_csv("/content/10707-project/data/speechfake_splits.csv")

# Filter dataframes by split
train_df = master_df[master_df['split'] == 'train'].reset_index(drop=True)
val_df = master_df[master_df['split'] == 'val'].reset_index(drop=True)
test_df = master_df[master_df['split'] == 'test'].reset_index(drop=True)

# Create Dataset objects
train_dataset = XMADDataset(train_df)
val_dataset = XMADDataset(val_df)
test_dataset = XMADDataset(test_df)

# Create DataLoaders
# Set num_workers to 2 or 4 to speed up loading on Colab
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Loaders created: {len(train_loader)} train batches, {len(val_loader)} val batches.")

In [ ]:
# args/configs
d_args = {
    "filts": [[1, 32], [32, 32], [32, 64], [64, 64], [64, 128]], # Example AASIST filter bank
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5],
    "temperatures": [2.0, 2.0, 1.0],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8  # Keep small for W2V2
lr = 0.0001
epochs = 10

In [ ]:
# wandb and huggingface setup
import wandb
from huggingface_hub import HfApi, login

# Initialize W&B
wandb.init(
    project="audio-deepfake-detection",
    config={
        "learning_rate": lr,
        "architecture": "W2V2_AASIST",
        "dataset": "SpeechFake",
        "epochs": epochs,
    }
)

# Login to HF (Run this once or use a token)
login()
api = HfApi()
repo_id = "Joel-10707-Project-S26/baseline-w2v2aasist"

In [ ]:
# import model and initialize optimizer
if "/content/10707-project/utils" not in sys.path:
    sys.path.append("/content/10707-project/baseline")
from w2v2_aasist import W2V2_AASIST

model = W2V2_AASIST(d_args).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [ ]:
# train
for epoch in range(epochs):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    loss, acc = validate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Loss {loss:.4f}, Val Acc {acc:.4f}")